In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="Delivery-Time-Analysis-Prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/Delivery-Time-Analysis-Prediction"

Repository mridul0010/Delivery-Time-Analysis-Prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow")

In [4]:
mlflow.set_experiment("Hyperparameter Tuning - XGB")

<Experiment: artifact_location='mlflow-artifacts:/b030b782ed11437b97153d6288f8e038', creation_time=1782935208842, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1782935208842, lifecycle_stage='active', name='Hyperparameter Tuning - XGB', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            ['Low', 'Medium', 'High', 'Jam'],                         # Road_traffic_density
            ['poor', 'Average', 'Good' , 'Excellent'],                # Vehicle_condition
            ['No', 'Yes'],                                            # Festival
            ['Low', 'Medium', 'High'],                                # delivery_rating_group
            ['Young', 'Adult', 'Senior'],                             # age_group
            ['Short Distance', 'Medium Distance', 'Long Distance']    # distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [15]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [16]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [17]:
pt = PowerTransformer()

In [18]:
def objective(trial):
    # Safe sequential execution or thread-isolated nested runs
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 1, 40),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
            "subsample": trial.suggest_float("subsample", 0.4, 1),
            "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
            "gamma": trial.suggest_float("gamma", 0, 10), 
            "reg_lambda": trial.suggest_float("reg_lambda", 0, 100),
            "random_state": 42,
            "n_jobs": -1,
        }

        mlflow.log_params(params)

        xgb = XGBRegressor(**params)
        model = TransformedTargetRegressor(regressor=xgb, transformer=pt)

        # REMOVED: model.fit(...) <- cross_val_score handles this internally
        cv_score = cross_val_score(
            model,
            X_train_trans, 
            y_train, 
            cv=10, 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())
        mlflow.log_metric("cross_val_error", mean_score)

        return mean_score

In [19]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # Fixed n_jobs=1 to avoid MLflow concurrent logging crashes
    study.optimize(objective, n_trials=50, n_jobs=1, show_progress_bar=True)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_score", study.best_value)

    # Re-train using the best parameters
    best_xgb = XGBRegressor(**study.best_params)
    best_model = TransformedTargetRegressor(regressor=best_xgb, transformer=pt)
    best_model.fit(X_train_trans, y_train)

    # Evaluate
    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5, 
        n_jobs=-1
    )

    mlflow.log_metric("Training_error", mean_absolute_error(y_train, y_pred_train))
    mlflow.log_metric("Test_error", mean_absolute_error(y_test, y_pred_test))
    mlflow.log_metric("Training_r2", r2_score(y_train, y_pred_train))
    mlflow.log_metric("Test_r2", r2_score(y_test, y_pred_test))
    mlflow.log_metric("cross_val", -scores.mean())

    # Generate and log plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    mlflow.sklearn.log_model(best_model, artifact_path="model_XGB")

[I 2026-07-02 20:28:28,047] A new study created in memory with name: no-name-4a166f7b-aa40-45ca-b3d1-b3968041fdae
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run fortunate-fly-727 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2/runs/7c7a0f66798b49449a519d866b7d01f0
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2


Best trial: 0. Best value: 3.28703:   2%|▏         | 1/50 [00:09<07:31,  9.21s/it]

[I 2026-07-02 20:28:38,605] Trial 0 finished with value: 3.287027907371521 and parameters: {'n_estimators': 288, 'max_depth': 16, 'learning_rate': 0.008315620217327391, 'subsample': 0.6716705959927927, 'min_child_weight': 10, 'gamma': 3.9860728493229947, 'reg_lambda': 50.56427404624914}. Best is trial 0 with value: 3.287027907371521.
🏃 View run brawny-crane-755 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2/runs/fa855a988c5a41dd849a9a40a7f43018
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2


Best trial: 0. Best value: 3.28703:   4%|▍         | 2/50 [00:15<05:46,  7.22s/it]

...

[I 2026-07-02 20:35:37,374] Trial 48 finished with value: 3.100641393661499 and parameters: {'n_estimators': 257, 'max_depth': 26, 'learning_rate': 0.08931926833395, 'subsample': 0.8384093244933899, 'min_child_weight': 12, 'gamma': 3.746669666025886, 'reg_lambda': 71.12442621547865}. Best is trial 27 with value: 2.998590660095215.
🏃 View run agreeable-goat-126 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2/runs/9b54ed2aac6a4a9fb4fe37521b11e700
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2


Best trial: 27. Best value: 2.99859: 100%|██████████| 50/50 [07:12<00:00,  8.65s/it]


[I 2026-07-02 20:35:42,027] Trial 49 finished with value: 3.00485520362854 and parameters: {'n_estimators': 237, 'max_depth': 29, 'learning_rate': 0.19398851765128425, 'subsample': 0.9604612677561407, 'min_child_weight': 10, 'gamma': 1.1700391321232355, 'reg_lambda': 31.08707552628585}. Best is trial 27 with value: 2.998590660095215.


2026/07/02 20:35:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/02 20:35:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2/runs/3df8ba5fe69d4bdd9c3a738211209ccf
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/2


In [20]:
study.best_value

2.998590660095215

In [21]:
study.best_params

{'n_estimators': 245,
 'max_depth': 23,
 'learning_rate': 0.19864909172366368,
 'subsample': 0.9180765181845632,
 'min_child_weight': 7,
 'gamma': 1.0407049003047555,
 'reg_lambda': 22.40266224283313}

In [22]:
best_xgb = XGBRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_xgb , transformer=pt)

model.fit(X_train_trans , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","XGBRegressor(...ree=None, ...)"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None


In [23]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8361227512359619

In [24]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [25]:
fig_history.show()

In [26]:
fig_parallel 

In [27]:
fig_importance

In [28]:
fig_slice